# Workflow Optimization

Optimization helps you systematically improve your agent's performance by tuning prompts and parameters. This tutorial covers NAT's optimization capabilities.

## What You'll Learn

1. Understanding optimization types
2. Configuring numeric optimization (hyperparameters)
3. Configuring prompt optimization
4. Running optimization via SDK
5. Running optimization via CLI
6. Applying optimized configurations

## Why Optimize?

- **Improve accuracy** - Find the best prompts and parameters
- **Reduce costs** - Optimize for efficiency
- **Automate tuning** - Replace manual trial-and-error
- **Data-driven decisions** - Use metrics to guide improvements


In [ ]:
import sys
from pathlib import Path

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


## Optimization Types

NAT supports two types of optimization:

### 1. Numeric Optimization
Optimize numerical parameters like:
- `temperature` - LLM randomness (0.0-1.0)
- `max_tokens` - Response length
- `top_p` - Nucleus sampling parameter

### 2. Prompt Optimization
Optimize text prompts:
- System prompts
- Additional instructions
- Few-shot examples


## Step 1: Create a Workflow to Optimize


In [ ]:
import json

from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM with initial parameters
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.5,      # We'll optimize this
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

# Create agent with system prompt (we'll optimize this too)
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    additional_instructions="Be concise in your responses.",  # Will optimize
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


## Step 2: Create Optimization Dataset

The optimizer needs a dataset to evaluate performance:


In [ ]:
# Create optimization dataset
opt_data = [
    {"input": "What is 2 + 2?", "expected_output": "4"},
    {"input": "What is 10 * 5?", "expected_output": "50"},
    {"input": "What is 100 / 4?", "expected_output": "25"},
    {"input": "What is 15 - 7?", "expected_output": "8"},
]

# Save dataset
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / "opt_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(opt_data, f, indent=2)

print(f"📊 Dataset saved to: {dataset_path}")


## Step 3: Configure the Optimizer

The optimizer requires:
1. **Evaluation metrics** - What to optimize for
2. **Target parameters** - What to tune
3. **Search space** - Range of values to explore


In [ ]:
from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.optimizer.config import NumericOptimization
from nat.optimizer.config import NumericParam
from nat.optimizer.config import OptimizerMetric
from nat.optimizer.config import PromptOptimization
from nat.optimizer.config import PromptParam
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation
from nat.utils.sdk.nat_optimizer import NatOptimizer

# Create evaluator for optimization metric
accuracy_eval = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy",
)

# Configure evaluation (required for optimization)
evaluation = NatEvaluation(
    output_dir=Path("./opt_results/eval"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    evaluators=[accuracy_eval],
)

workflow.add_evaluator(evaluation)
print("✅ Evaluator configured")


In [ ]:
# Configure optimizer
optimizer = NatOptimizer(
    output_path=Path("./opt_results"),

    # What metric to optimize
    eval_metrics=[
        OptimizerMetric(
            name="accuracy",           # Must match evaluator name
            target=1.0,                # Target value
            maximize=True,             # Higher is better
        )
    ],

    # Numeric parameters to optimize
    numeric=NumericOptimization(
        params=[
            NumericParam(
                config_path="llms.nim_llm.temperature",  # Path in config
                min_value=0.0,                            # Minimum value
                max_value=1.0,                            # Maximum value
            ),
        ],
        n_trials=5,  # Number of optimization trials
    ),

    # Prompt parameters to optimize (optional)
    prompt=PromptOptimization(
        params=[
            PromptParam(
                config_path="workflow.additional_instructions",
                initial_value="Be concise in your responses.",
                n_candidates=3,  # Number of prompt variants to try
            ),
        ],
    ),

    # Number of evaluations per parameter set
    reps_per_param_set=1,
)

workflow.add_optimizer(optimizer)
print("✅ Optimizer configured")


## Step 4: Save the Configuration


In [ ]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "opt_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION WITH OPTIMIZER:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


## Step 5: Run Optimization

### Via Python SDK


In [ ]:
# Run optimization (uncomment to execute)
# await workflow.optimize()
# print("✅ Optimization complete! Check ./opt_results for results")


### Via CLI

```bash
# Run optimization
nat optimize --config_file configs/opt_workflow.yaml

# With custom dataset
nat optimize --config_file configs/opt_workflow.yaml \
    --dataset data/opt_dataset.json

# Override output path
nat optimize --config_file configs/opt_workflow.yaml \
    --override optimizer.output_path=./custom_results
```


## Understanding Optimization Results

After optimization, you'll find:

```
opt_results/
├── best_config.yaml        # Optimized configuration
├── optimization_history.json  # All tried configurations
├── eval/                   # Evaluation results for each trial
│   ├── trial_0/
│   ├── trial_1/
│   └── ...
└── optimization_log.txt    # Detailed log
```

### Example Optimization History
```json
{
  "best_trial": {
    "temperature": 0.1,
    "additional_instructions": "Be precise and show your work.",
    "accuracy": 0.95
  },
  "trials": [
    {"temperature": 0.5, "accuracy": 0.75},
    {"temperature": 0.1, "accuracy": 0.95},
    {"temperature": 0.8, "accuracy": 0.60}
  ]
}
```


## Summary

In this tutorial, you learned:

✅ Understanding numeric and prompt optimization  
✅ Configuring `NatOptimizer` with metrics and parameters  
✅ Defining search spaces for optimization  
✅ Running optimization via SDK and CLI  
✅ Understanding optimization results  

## Next Steps

- **[12_observability.ipynb](./12_observability.ipynb)** - Add tracing and monitoring
- **[08_configuration_guide.ipynb](./08_configuration_guide.ipynb)** - Deep dive into YAML configuration
